# WR MLP - Quantile Regression

Umesto klasicnog point estimate (MSE/Huber), treniramo MLP da predvidja **tri kvantila istovremeno**: q10, q50, q90, koristeci **pinball loss**.

Prednosti:
- **q50 (medijana) kao point prediction** je robusniji na outliere od proseka - cesto bolji MAE
- **Besplatne intervali neizvesnosti** (q90 - q10) - za NFL "boom or bust" utakmice ovo je dragoceno
- Model uci pravi oblik distribucije, ne samo jednu tacku

Arhitektura: ista kao MLP-B (5 slojeva [448, 128, 320, 448, 256], LayerNorm, dropout 0.5), ali sa **3-izlaznom glavom** i pinball loss-om.

Target: sqrt(yards), invert kroz `pred**2` na kraju.


---
## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import os, json, time, random

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers, Model, Input
from tensorflow.keras.layers import Dense, Dropout, LayerNormalization

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)
print(f'TensorFlow {tf.__version__}')


---
## 2. Load Top-40 Features & Data

In [ ]:
df = pd.read_csv('../data/fully combined/wr_all_weeks.csv')
df['week'] = df['game_id'].str.split('_').str[1].astype(int)

# Load the top 40 feature list from feature selection notebook
with open('../results/selected_features_top40.json', 'r') as f:
    feat_info = json.load(f)
feat_cols = feat_info['selected_features']
print(f'Loaded {len(feat_cols)} features from selected_features_top40.json')

# Basic temporal features used in comparison notebook
# (lag1, roll5 etc. should already exist in wr_all_weeks.csv from feature engineering)
missing = [c for c in feat_cols if c not in df.columns]
if missing:
    print(f'Missing columns in base csv: {missing[:5]}... ({len(missing)} total)')
    print('Falling back to recomputing like MLP comparison notebook expects.')
else:
    print('All features present.')


---
## 3. Recreate Top-40 Features (same pipeline as MLP comparison)


In [ ]:
# Recreate the same pipeline as in WR_MLP_Comparison notebook
df = df.sort_values(['receiver_player_id', 'season', 'week']).reset_index(drop=True)

# Rolling/lag feature engineering to match the 184-feature space the selector ran on
base_stats = [
    'receiving_yards', 'targets', 'receptions', 'air_yards', 'yac',
    'first_downs', 'tds', 'epa', 'wpa', 'catch_rate', 'yards_per_target',
    'avg_depth', 'adot', 'success_rate', 'target_share', 'air_yard_share',
    'target_share_std', 'reception_std', 'team_pass_attempts', 'team_air_yards',
    'team_epa', 'qb_completions', 'qb_attempts', 'qb_comp_pct', 'qb_cpoe',
    'avg_score_diff', 'wp_var', 'yards_Q1', 'yards_Q2', 'yards_Q3',
    'def_yards_dev', 'def_epa_dev',
]

g = df.groupby('receiver_player_id', group_keys=False)
for col in base_stats:
    if col in df.columns:
        df[f'{col}_lag1'] = g[col].shift(1)
        df[f'{col}_roll5'] = g[col].shift(1).rolling(5, min_periods=1).mean().reset_index(level=0, drop=True)

# Fill NaN for selected feature columns
available_feats = [c for c in feat_cols if c in df.columns]
still_missing = [c for c in feat_cols if c not in df.columns]
if still_missing:
    print(f'Still missing after engineering: {still_missing}')
    raise ValueError('Feature pipeline mismatch - inspect selected_features_top40.json')

df[available_feats] = df[available_feats].fillna(0)
print(f'All {len(available_feats)} selected features ready.')
feat_cols = available_feats


---
## 4. Temporal Split & Scaling

In [ ]:
train_seasons = list(range(2015, 2022))
val_seasons = [2022, 2023]
test_seasons = [2024, 2025]

tr_mask = df['season'].isin(train_seasons)
va_mask = df['season'].isin(val_seasons)
te_mask = df['season'].isin(test_seasons)

X_tr_raw = df.loc[tr_mask, feat_cols].values
X_va_raw = df.loc[va_mask, feat_cols].values
X_te_raw = df.loc[te_mask, feat_cols].values
y_tr = df.loc[tr_mask, 'receiving_yards'].values.astype(np.float32)
y_va = df.loc[va_mask, 'receiving_yards'].values.astype(np.float32)
y_te = df.loc[te_mask, 'receiving_yards'].values.astype(np.float32)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr_raw).astype(np.float32)
X_va = scaler.transform(X_va_raw).astype(np.float32)
X_te = scaler.transform(X_te_raw).astype(np.float32)

# Target transformation: sqrt
y_tr_s = np.sqrt(np.clip(y_tr, 0, None)).astype(np.float32)
y_va_s = np.sqrt(np.clip(y_va, 0, None)).astype(np.float32)
y_te_s = np.sqrt(np.clip(y_te, 0, None)).astype(np.float32)

print(f'train={len(y_tr)}  val={len(y_va)}  test={len(y_te)}')
print(f'Features: {X_tr.shape[1]}')


---
## 5. Pinball Loss for Multiple Quantiles


In [ ]:
QUANTILES = [0.1, 0.5, 0.9]
N_Q = len(QUANTILES)

def multi_pinball_loss(quantiles):
    q_tensor = tf.constant(quantiles, dtype=tf.float32)  # (N_Q,)
    def loss(y_true, y_pred):
        # y_true: (batch, 1)  y_pred: (batch, N_Q)
        y_true = tf.reshape(y_true, (-1, 1))  # (batch, 1)
        e = y_true - y_pred                   # (batch, N_Q)
        return tf.reduce_mean(tf.maximum(q_tensor * e, (q_tensor - 1) * e))
    return loss

# MAE of the median prediction - what we care about at inference
def median_mae(y_true, y_pred):
    y_true = tf.reshape(y_true, (-1, 1))
    median = y_pred[:, 1:2]
    return tf.reduce_mean(tf.abs(y_true - median))

print(f'Quantiles: {QUANTILES}')


---
## 6. MLP-B Architecture with 3-Output Head


In [ ]:
def build_quantile_mlp(n_features, n_quantiles=3):
    units = [448, 128, 320, 448, 256]
    dropout = 0.5
    lr = 0.0037
    wd = 1.5e-5

    inp = Input(shape=(n_features,))
    x = inp
    for u in units:
        x = Dense(u, activation='relu')(x)
        x = LayerNormalization()(x)
        x = Dropout(dropout)(x)
    out = Dense(n_quantiles)(x)
    model = Model(inp, out)

    opt = optimizers.AdamW(learning_rate=lr, weight_decay=wd)
    model.compile(optimizer=opt, loss=multi_pinball_loss(QUANTILES), metrics=[median_mae])
    return model

_m = build_quantile_mlp(X_tr.shape[1])
_m.summary()


---
## 7. Train

In [ ]:
def make_weights(strength, y_orig):
    mean_y = np.mean(np.clip(y_orig, 0, None))
    return 1.0 + strength * np.sqrt(np.clip(y_orig, 0, None) / mean_y)

sw_tr = make_weights(1.0, y_tr)

tf.keras.backend.clear_session()
tf.random.set_seed(SEED); np.random.seed(SEED); random.seed(SEED)

model = build_quantile_mlp(X_tr.shape[1])

cb = [
    callbacks.EarlyStopping(monitor='val_loss', patience=25,
                            restore_best_weights=True, min_delta=1e-4),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=8, min_lr=1e-6),
]

t0 = time.time()
hist = model.fit(
    X_tr, y_tr_s,
    sample_weight=sw_tr,
    validation_data=(X_va, y_va_s),
    epochs=300, batch_size=32,
    callbacks=cb, verbose=1,
)
print(f'\nTrained in {time.time()-t0:.0f}s')


---
## 8. Evaluation - Point Metrics (using q50) & Quantile Coverage


In [ ]:
pred_te = model.predict(X_te, verbose=0)  # (N, 3) in sqrt space
q10_sqrt = pred_te[:, 0]
q50_sqrt = pred_te[:, 1]
q90_sqrt = pred_te[:, 2]

# Invert sqrt transform
q10 = np.clip(q10_sqrt, 0, None) ** 2
q50 = np.clip(q50_sqrt, 0, None) ** 2
q90 = np.clip(q90_sqrt, 0, None) ** 2

# Fix crossing quantiles (rare but possible)
q10_fix = np.minimum(q10, q50)
q90_fix = np.maximum(q90, q50)

mae_q50 = mean_absolute_error(y_te, q50)
rmse_q50 = np.sqrt(mean_squared_error(y_te, q50))
r2_q50 = r2_score(y_te, q50)

coverage = np.mean((y_te >= q10_fix) & (y_te <= q90_fix))
interval_width = np.mean(q90_fix - q10_fix)

print(f'=== Quantile MLP (q50 as point prediction) ===')
print(f'  MAE:  {mae_q50:.2f}')
print(f'  RMSE: {rmse_q50:.2f}')
print(f'  R2:   {r2_q50:.4f}')
print(f'\n=== Uncertainty Interval (q90 - q10) ===')
print(f'  Coverage (target 80%): {coverage*100:.1f}%')
print(f'  Mean interval width:   {interval_width:.1f} yards')

print(f'\n=== Comparison ===')
print(f'  MLP-B (point estimate): MAE=18.19  RMSE=25.23  R2=0.3289')
print(f'  MLP Quantile (q50):     MAE={mae_q50:.2f}  RMSE={rmse_q50:.2f}  R2={r2_q50:.4f}')


---
## 9. Visualize Predictions + Intervals

In [ ]:
# Plot some samples with their q10-q90 intervals
np.random.seed(0)
idx = np.random.choice(len(y_te), 80, replace=False)
idx = idx[np.argsort(y_te[idx])]

fig, ax = plt.subplots(figsize=(14, 6))
xs = np.arange(len(idx))
ax.fill_between(xs, q10_fix[idx], q90_fix[idx], alpha=0.3, color='steelblue', label='q10-q90 interval')
ax.plot(xs, q50[idx], 'o-', color='steelblue', markersize=4, label='q50 prediction')
ax.plot(xs, y_te[idx], 'x', color='red', markersize=7, label='True')
ax.set_xlabel('Sample (sorted by true yards)')
ax.set_ylabel('Receiving yards')
ax.set_title(f'Quantile MLP predictions vs truth (coverage={coverage*100:.1f}%)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/mlp_quantile_predictions.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 10. Learning Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(hist.history['loss']) + 1)
axes[0].plot(ep, hist.history['loss'], label='Train')
axes[0].plot(ep, hist.history['val_loss'], label='Val')
axes[0].set_title('Pinball Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, hist.history['median_mae'], label='Train')
axes[1].plot(ep, hist.history['val_median_mae'], label='Val')
axes[1].set_title('Median MAE (sqrt space)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/mlp_quantile_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 11. Save Results

In [ ]:
output = {
    'quantiles': QUANTILES,
    'test_metrics': {
        'mae_q50': float(mae_q50),
        'rmse_q50': float(rmse_q50),
        'r2_q50': float(r2_q50),
        'coverage_80pct': float(coverage),
        'mean_interval_width': float(interval_width),
    },
    'best_epoch': int(np.argmin(hist.history['val_loss']) + 1),
    'n_features': int(X_tr.shape[1]),
    'architecture': {
        'units': [448, 128, 320, 448, 256],
        'dropout': 0.5, 'lr': 0.0037, 'wd': 1.5e-5,
    },
    'comparison': {
        'MLP-B (point)':  {'mae': 18.19, 'rmse': 25.23, 'r2': 0.3289},
        'MLP-Quantile':   {'mae': round(mae_q50, 2), 'rmse': round(rmse_q50, 2), 'r2': round(r2_q50, 4)},
    },
}

with open('../results/mlp_quantile_results.json', 'w') as f:
    json.dump(output, f, indent=2, default=str)

print('Saved: results/mlp_quantile_results.json')
